# Finetuning GPT-2 small

GPT-2 (small):  

- 117 million parameters
- 12 layers
- Hidden size: 768
- 12 attention heads

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
import torch
from transformers import pipeline

C:\Users\micha\anaconda3\envs\transformers\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from datasets import load_dataset, DatasetDict
from transformers import AutoTokenizer

# Load the 'vicclab/fairy_tales' dataset from Hugging Face
# This dataset contains fairy tale texts which can be used for fine-tuning a language model
dataset = load_dataset('vicclab/fairy_tales')

In [3]:
train_val = dataset["train"].train_test_split(
    test_size=0.2, seed=42)

In [4]:
dataset = DatasetDict({
    "train": train_val["train"],
    "validation": train_val["test"]
})

In [5]:
print("Train size:", len(dataset["train"]))
print("Validation size:", len(dataset["validation"]))

Train size: 82878
Validation size: 20720


In [6]:
# Load the tokenizer associated with GPT-2
# Tokenizer converts text into token IDs that the model can process
tokenizer = AutoTokenizer.from_pretrained('openai-community/gpt2') 

# GPT-2 does not have a dedicated padding token by default
# Set the padding token to the end-of-sequence (EOS) token
# This is necessary for batch processing during training
tokenizer.pad_token = tokenizer.eos_token

In [7]:
def tokenize_function(examples):
    """
    Convert raw text into token IDs suitable for GPT-2 model input.
    
    Args:
        examples (dict): A batch of examples from the dataset, 
                         each example should have a "text" field.
                         
    Returns:
        dict: A dictionary containing tokenized inputs, ready for model training.
    """

    # Tokenize the 'text' field
    # - truncation = True ensures that sequences longer than max_length are cut off
    # - max_length = 256 limits each sequence to 256 tokens (important for memory and model constraints)
    enc = tokenizer(
        examples["text"],
        truncation=True,
        max_length=256
    )
    return enc

In [8]:
# Apply the tokenization function to the whole dataset
tokenized_datasets = dataset.map(
    tokenize_function,                              # function to convert text into token IDs
    batched=True,                                   # process multiple examples at once for efficiency
    remove_columns=dataset["train"].column_names    # remove original text columns, keep only tokenized inputs
)

Map: 100%|█████████████████████████████████████████████████████████████| 20720/20720 [00:00<00:00, 36480.77 examples/s]


In [9]:
# Filter out any examples that ended up empty after tokenization
# (e.g., texts that were completely blank or got truncated to zero length)
tokenized_datasets = tokenized_datasets.filter(
    lambda x: len(x["input_ids"]) > 0
)

Filter: 100%|██████████████████████████████████████████████████████████| 20720/20720 [00:00<00:00, 62739.98 examples/s]


In [10]:
from transformers import DataCollatorForLanguageModeling

# Data collators are responsible for dynamically batching and preparing inputs during training
# For causal language modeling (like GPT-2), we don't use masked language modeling (MLM)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False            # GPT-2 uses causal LM, not masked LM
)

# The data collator will:
# - pad sequences in a batch to the same length
# - create 'labels' that match the input_ids for next-token prediction

In [11]:
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer


# Load the GPT-2 model for causal language modeling
# - "openai-community/gpt2" is the pre-trained model
# - .to("cuda") moves the model to the GPU for faster training
model = AutoModelForCausalLM.from_pretrained(
    "openai-community/gpt2"
).to("cuda")

training_args = TrainingArguments(
    output_dir="models/results",        # directory to save model checkpoints
    eval_strategy="epoch",              # evaluate the model at the end of each epoch
    num_train_epochs=5,                 # total number of training epochs       
    per_device_train_batch_size=4,      # batch size per GPU/CPU for training
    per_device_eval_batch_size=4,       # batch size per GPU/CPU for evaluation
    learning_rate=5e-5,                 # initial learning rate
    warmup_ratio=0.1,                   # fraction of steps used for learning rate warmup
    weight_decay=0.01,                  # L2 weight decay for regularization
    logging_dir="models/logs",          # directory for storing logs
    save_strategy="epoch",              # save checkpoints at the end of each epoch
    report_to="none"                    # disable reporting to external tools like WandB
)

In [15]:
# Create a Hugging Face Trainer object
# The Trainer handles the training loop, evaluation, checkpointing, and logging
trainer = Trainer(
    model=model,                                     # the GPT-2 model to fine-tune
    args=training_args,                              # training hyperparameters defined earlier
    train_dataset=tokenized_datasets["train"],       # training dataset
    eval_dataset=tokenized_datasets["validation"],   # validation dataset for evaluation
    data_collator=data_collator,                     # handles batching, padding, and labels
)

# Start the fine-tuning process
# This will:
# - loop over the training dataset for the specified number of epochs
# - evaluate the model on the validation set at the end of each epoch
# - save checkpoints in the output directory
trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,3.412700,3.552787
2,3.177700,3.476668
3,2.904400,3.489201
4,2.717600,3.526197
5,2.532700,3.583601


TrainOutput(global_step=83275, training_loss=2.977680019079762, metrics={'train_runtime': 10073.2578, 'train_samples_per_second': 33.067, 'train_steps_per_second': 8.267, 'total_flos': 3373994603520000.0, 'train_loss': 2.977680019079762, 'epoch': 5.0})

---